<a href="https://colab.research.google.com/github/tor-apiwit/var_risk_engine/blob/main/var_risk_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Portfolio Value at Risk (VaR) Engine

Computes portfolio VaR using three methods — **Parametric (Variance-Covariance)**, **Historical**, and **Monte Carlo** — finds the portfolio weights that minimize risk / minimize tail loss via constrained optimization, and runs a rolling 1-year backtest to see how VaR changes over time.

This notebook is a thin orchestration layer: all calculation logic lives in the `src/` package (`config`, `data_loader`, `portfolio_stats`, `var_models`, `optimization`, `backtest`, `reporting`). The notebook itself just wires those modules together and displays results.

## 1. Setup

If running in Colab, clone the repo (or otherwise ensure `src/` is on the Python path) so the project modules can be imported.

In [ ]:
import sys
sys.path.append("src")  # make the src/ package importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.data_loader import load_close_prices, compute_returns
from src.portfolio_stats import scale_mu, scale_cov
from src.var_models import performance_portfolio, historical_var, monte_carlo_var
from src.optimization import find_min_risk_portfolio, find_min_loss_portfolio
from src.backtest import build_rolling_windows, compute_actual_return, calculate_var_metrics, kupiec_test
from src.reporting import show_portfolio, show_var_comparison, plot_rolling_var

## 2. Config

All constants are defined in `src/config.py` rather than scattered as "magic numbers" throughout the notebook, so a single parameter change propagates everywhere it's used.

**Scaling convention** (used consistently throughout): volatility scales with `sqrt(horizon)`, variance/covariance scales with `horizon` — consistent with the i.i.d. daily returns assumption.

In [ ]:
print("Tickers:", config.TICKERS)
print("Date range:", config.START_DATE, "to", config.END_DATE)
print("Initial weights:", config.W_PORT_INITIAL)
print("Investment horizon (days):", config.INVESTMENT_HORIZON)
print("Confidence level:", config.CONFIDENCE)
print("Rolling window (days):", config.ROLLING_WINDOW)
print("Number of rolling windows:", config.N_ROLLING_WINDOWS)

## 3. Load Price Data and Compute Returns

**`load_close_prices`**: fetches each ticker's daily close price from Yahoo Finance and merges them into a single DataFrame.

Each ticker is wrapped in its own try/except — if a given ticker cannot be fetched (e.g. delisted, or rate-limited), the code prints a warning and skips it instead of crashing the whole script.

**`compute_returns`**: computes
- **Daily log return**: `ln(P_t / P_{t-1})` — log returns are used because they are additive across time.
- **Annual simple return**: uses year-end close prices, simple return `(P_t/P_{t-1}) - 1`, to show a yearly overview of performance.

In [ ]:
close_prices = load_close_prices(config.TICKERS, config.START_DATE, config.END_DATE)
returns_daily, returns_annual = compute_returns(close_prices)
returns_daily

## 4. Portfolio Statistics (Consistent Scaling)

`scale_mu` and `scale_cov` (from `src/portfolio_stats.py`) are the single place where mean and covariance are scaled from daily to the desired horizon — used consistently everywhere in the project instead of writing `* investment_horizon` in multiple scattered places (a source of inconsistency in earlier versions of this codebase).

- Mean scales linearly with horizon (`mu * horizon`)
- Covariance scales linearly with horizon as well (`cov * horizon`), because Var(n-day return) = n × Var(1-day return) under the i.i.d. returns assumption.

In [ ]:
mu_port = scale_mu(returns_daily.mean(), config.INVESTMENT_HORIZON)
sigma_port = scale_cov(returns_daily.cov(), config.INVESTMENT_HORIZON)
corr_port = returns_daily.corr()

## 5. Parametric VaR (Variance-Covariance Method)

`performance_portfolio` (from `src/var_models.py`) computes:
- **Portfolio return**: `w @ mu`
- **Portfolio risk (std)**: `sqrt(w @ sigma @ w)`
- **VaR**: `return + z * risk`, where `z = norm.ppf(1 - confidence)` is the left-tail quantile of the standard normal. E.g. confidence = 0.95 -> `z = norm.ppf(0.05) ≈ -1.645`

## 6. Portfolio Optimization

Two functions (from `src/optimization.py`) find portfolio weights via `scipy.optimize.minimize` (SLSQP), subject to: weights sum to 1, and each asset's weight lies within `[min_weight, max_weight]`.

**`find_min_risk_portfolio`** — Global Minimum-Variance Portfolio
- Objective: minimize `sqrt(w @ sigma @ w)` only (expected return is ignored entirely).

**`find_min_loss_portfolio`** *(previously named `find_min_var_portfolio` — renamed to reduce confusion)*
- VaR in this project is stored as a **negative** value (more negative = worse loss). So "maximizing VaR" mathematically is the same as "minimizing the magnitude of the tail loss". The old name `find_min_var_portfolio` made this easy to misread as "minimizing VaR" (which would actually mean approaching -∞, the worst outcome) — hence the rename to `find_min_loss_portfolio`, which matches what the function actually does.
- Objective: minimize `-(return + z * risk)` ⟺ maximize `(return + z * risk)` ⟺ minimize tail loss

## 7. Reporting Helpers

Display helpers (from `src/reporting.py`), kept separate from calculation logic (separation of concerns) so the logic can be tested/modified without affecting display, and vice versa.

## 8. Run Parametric VaR

Compares three portfolios:
1. **initial** — the manually specified starting weights (`W_PORT_INITIAL`)
2. **minrisk** — the weights that minimize risk (portfolio std), ignoring return
3. **minloss** — the weights that minimize tail loss (VaR), accounting for both return and risk via the VaR formula

In [ ]:
output_initial = performance_portfolio(config.W_PORT_INITIAL, mu_port, sigma_port, config.CONFIDENCE)
output_minrisk = find_min_risk_portfolio(mu_port, sigma_port, config.W_PORT_INITIAL, 0, 1, config.CONFIDENCE)
output_minloss = find_min_loss_portfolio(mu_port, sigma_port, config.W_PORT_INITIAL, 0, 1, config.CONFIDENCE)

show_portfolio("output_initial", output_initial, config.TICKERS)
show_portfolio("output_minrisk", output_minrisk, config.TICKERS)
show_portfolio("output_minloss", output_minloss, config.TICKERS)

## 9. Historical VaR

Finds VaR from the **empirical quantile** of actual historical returns, without assuming returns are normally distributed.

In [ ]:
show_var_comparison(
    "VaR Historical (%)",
    {
        "initial": historical_var(returns_daily, output_initial["portfolio_weight"], 1 - config.CONFIDENCE),
        "minrisk": historical_var(returns_daily, output_minrisk["portfolio_weight"], 1 - config.CONFIDENCE),
        "minloss": historical_var(returns_daily, output_minloss["portfolio_weight"], 1 - config.CONFIDENCE),
    },
)

## 10. Monte Carlo VaR

Simulates correlated log-returns using **Cholesky decomposition** of the correlation matrix, then finds the quantile of the **simulated portfolio**.

Steps:
1. `L = cholesky(corr)` — decompose the correlation matrix to generate correlated shocks
2. Draw `Z_independent` as independent standard normals, then transform to `Z_correlated = Z_independent @ L.T`
3. Simulate each asset's log return via the GBM formula: `(mu - 0.5*sigma^2) + sigma*Z_correlated`
4. Combine into the portfolio return: `log_returns @ w_port`
5. Take the quantile of the **simulated portfolio** (not the quantile of each individual asset)

In [ ]:
mu_daily = returns_daily.mean().to_numpy()
sigma_daily = returns_daily.std().to_numpy()
corr_matrix = corr_port.to_numpy()

mc_kwargs = dict(
    mu_daily=mu_daily,
    sigma_daily=sigma_daily,
    corr=corr_matrix,
    horizon=config.INVESTMENT_HORIZON,
    alpha=1 - config.CONFIDENCE,
    n_sims=config.MC_SIMS_STATIC,
)

show_var_comparison(
    "VaR Monte Carlo (%)",
    {
        "initial": monte_carlo_var(w_port=output_initial["portfolio_weight"], **mc_kwargs),
        "minrisk": monte_carlo_var(w_port=output_minrisk["portfolio_weight"], **mc_kwargs),
        "minloss": monte_carlo_var(w_port=output_minloss["portfolio_weight"], **mc_kwargs),
    },
)

## 11. Rolling 1-Year Re-estimation

Recomputes all three VaR methods on a **rolling 252-day window** (~1 trading year), stepped by 1 day at a time, to see how VaR changes over time (e.g. VaR should worsen during periods of high market volatility).

**Index convention**: key `0` = the most recent window (today, looking back 252 days), key `-i` = the window shifted back `i` days from today.

> **Note on look-ahead bias**: `build_rolling_windows` prevents look-ahead bias *within each window's mu/sigma estimate* — window `-i` only uses data up through the day before `-i`. However, `compute_actual_return` (next section) applies a single fixed portfolio weight vector across the entire backtest. If that weight vector was derived from optimizing over the full sample (as `output_initial["portfolio_weight"]` is, further down), the backtest still carries a portfolio-level look-ahead bias distinct from the window-level protection here. Treat this as a caveat on the backtest's validity, not a solved problem — see the docstring in `src/backtest.py` for detail.

In [ ]:
returns_daily_1y = build_rolling_windows(returns_daily, config.ROLLING_WINDOW, config.N_ROLLING_WINDOWS)

In [ ]:
actual_return = compute_actual_return(returns_daily, output_initial["portfolio_weight"], config.N_ROLLING_WINDOWS)
output_rolling = calculate_var_metrics(
    returns_daily_1y,
    output_initial["portfolio_weight"],
    config.N_ROLLING_WINDOWS,
    config.CONFIDENCE,
    t=1,
    mc_sims=config.MC_SIMS_ROLLING,
)

result_var = pd.DataFrame({
    "Actual Return": actual_return["actual_return"],
    "Parametric VaR": output_rolling["Parametric VaR"],
    "Historical VaR": output_rolling["Historical VaR"],
    "Monte VaR": output_rolling["Monte VaR"]
})

result_var

In [ ]:
# Combined plot
plot_rolling_var(result_var)

## 12. Validating VaR Models with Kupiec's POF Test (Backtesting)

The `kupiec_test` function (from `src/backtest.py`) performs **backtesting** to evaluate how accurately each of the three VaR models (Parametric, Historical, and Monte Carlo) predicts risk, using the **Kupiec Proportion of Failures (POF) test**, based on the binomial distribution.

---

### 1. Statistical formulation

The test asks whether the **number of times the actual loss was worse than the model's predicted VaR (exceptions, $x$)** is consistent, in a statistically significant sense, with the stated confidence level ($c$).

*   **Null hypothesis ($H_0$):** the model is accurate — the observed exception rate ($p_{hat}$) equals the theoretical rate ($p = 1 - c$)
*   **Alternative hypothesis ($H_1$):** the model is inaccurate (it may underestimate or overestimate risk)

The Likelihood Ratio test statistic ($LR_{pof}$) is:
$$LR_{pof} = -2 \ln \left[ \frac{(1-p)^{N-x} p^x}{(1-\hat{p})^{N-x} \hat{p}^x} \right]$$

---

### 2. Line-by-line breakdown

*   **`x = np.sum(result_test["Actual Return"] < result_test["Model VaR"])`**
    Counts how many times `Actual Return` was less than `Model VaR` (i.e. the actual loss was worse than predicted, since VaR is stored as a negative value in this project)
*   **`p = 1 - confidence`**
    The theoretical exception rate (significance level, $\alpha$). E.g. at 95% confidence ($0.95$), $p$ equals 5% ($0.05$)
*   **`p_hat = x / N`**
    The observed exception rate from the data
*   **`lr_pof = -2 * np.log(...)`**
    The Likelihood Ratio statistic, compared against a Chi-Square distribution with 1 degree of freedom
*   **`Crit = chi2.ppf(1 - p, df=1)`**
    The critical value at significance level $\alpha = p = 1-\text{confidence}$, computed as `chi2.ppf(1-p, df=1)` so it matches the theoretical definition of critical value ($\alpha$-level) directly — reducing the chance that a future reader/editor misinterprets it
*   **`p_value = 1 - chi2.cdf(lr_pof, df=1)`**
    The p-value, used to decide between the hypotheses

---

### 3. Decision rule

Compare **the p-value against the significance level $\alpha$** (not against confidence), so the result maps directly onto standard statistical language without requiring inference:

| Statistical result | Decision | What it means for the model |
| :--- | :--- | :--- |
| **p-value $\ge \alpha$** <br>(equivalently $LR_{pof} \le$ Critical Value) | **Fail to reject $H_0$** | **Model passes:** the number of exceptions is within a statistically acceptable range |
| **p-value $< \alpha$** <br>(equivalently $LR_{pof} >$ Critical Value) | **Reject $H_0$** | **Model fails:** it may underestimate risk (if $x$ is too high) or overestimate risk (if $x$ is too low) |

---
### 4. Application in the following cells
The code compares the results of the three main models, so a risk manager can select the model that best reflects actual risk under the market conditions of that rolling window period.

In [ ]:
# Parametric VaR
kupiec_test(output_rolling["Parametric VaR"], actual_return["actual_return"], config.CONFIDENCE)

In [ ]:
# Historical VaR
kupiec_test(output_rolling["Historical VaR"], actual_return["actual_return"], config.CONFIDENCE)

In [ ]:
# Monte Carlo Simulation
kupiec_test(output_rolling["Monte VaR"], actual_return["actual_return"], config.CONFIDENCE)